# Lesson 18: How SmolVLA uses conditions

This lesson uses small code cells to show the real SmolVLA path. Run cells in order. The learner notebook has exercises for you to complete.

**Result:** a measurable account of how the VLM prefix, key/value cache, state, time, and action expert condition each prediction.

**Notebook pair:** [Reference answers](18-smolvla-conditioning-reference.ipynb).

**Evidence boundary:** CPU checks show tensor mechanics. They do not show a useful robot policy. Cells with an experiment tag load local model or LIBERO assets and can take much longer.

In [ ]:
import os
import sys
from pathlib import Path


def find_workspace() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")


WORKSPACE = find_workspace()
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
from collections.abc import Mapping

import torch
from torch.utils.data import default_collate

from vla_lab.teaching.act_model import frame_offset_seconds, open_libero_dataset
from vla_lab.teaching.interfaces import VLAInputs
from vla_lab.teaching.smolvla import SmolVLAAdapter, load_prepared_smolvla

## Model map: prefix attention and action conditioning

The prefix combines image patches, text tokens, and robot state. The VLM processes this prefix once and can store **key/value cache** tensors. In attention, a query compares with keys to make weights; those weights combine values. For one query, attention uses `softmax(Q Kᵀ / √d) V`: `Q` is the query, `K` the keys, `V` the values, and `d` the number of values per attention head. If two key scores are `0` and `ln(3)`, the weights are `0.25` and `0.75`; values `0` and `4` give context `3`. The cache stores prefix keys and values so each denoising step can reuse visual and language context. A cache belongs to one observation. Rebuild it after the camera image, instruction, or state changes.

The action expert is not the ACT decoder from earlier lessons. It is a small language-model-style stack paired with the VLM. It reads cached prefix keys/values and current action tokens. SmolVLA embeds the noisy action and the time `t` together in its suffix. Robot state is embedded with the observation prefix. These are two separate conditioning routes:

```text
(image patches, text tokens, state) ── VLM ── prefix K,V cache ───────────┐
                                                                         ├─ expert attention ─ velocity
(noisy action x_t, time t) ── action/time embedding ── expert query ─────┘
```

Shapes: prefix embeddings `(B,Lp,C)`; cached keys/values are per layer with batch `B`, prefix length `Lp`, and head dimensions; action embeddings `(B,T,C)`; time has one scalar per example `(B,)`. Attention masks prevent padded image/text tokens from contributing. The cache is not a compressed image: it is the attention state needed to reuse this prefix. For the two-key example above, the attention context is `0.25×0 + 0.75×4 = 3`; changing the state can change the prefix key/value vectors and therefore this context.

A useful architecture probe changes one condition at a time and measures a velocity difference. This does not prove policy quality. It shows which input route reaches the supplied model. Use identical action noise and time for both measurements so only the chosen condition changes.

## Exercise 18.1 — Measure a conditioning route

**Purpose.** Write a small probe that changes robot-state conditioning while keeping image, text, noisy actions, and time fixed. The model output should change when its state input changes.

Implement `state_condition_delta(adapter, inputs, x_t, time)`. It must return the mean absolute difference between (1) the predicted velocity with the supplied state and (2) the predicted velocity with an all-zero state tensor of the same shape, device, and dtype. Use `adapter.velocity` for both predictions. Do not change the model weights, mask, noise, or time.

A zero result can be valid for a random model if the selected layer is insensitive. A non-zero result proves only that state can affect the vector field. It does not prove that the policy uses state correctly on a robot.

In [ ]:
def state_condition_delta(
    adapter: SmolVLAAdapter, inputs: VLAInputs, x_t: torch.Tensor, time: torch.Tensor
) -> torch.Tensor:
    # TODO: compare predicted velocity with the supplied state and a zero-state copy.
    # Return mean absolute difference as a scalar tensor.
    raise NotImplementedError("Measure state conditioning")


# Do not change images, tokens, action noise, time, or model weights in this probe.

The CPU check uses a deterministic toy velocity function. The tagged model probe loads the published policy and compares state-on with zero-state using the same image, prompt, noise, and time.

In [ ]:
# Pure tensor check for the expected output contract.
class ToyAdapter:
    def velocity(
        self: "ToyAdapter",
        inputs: Mapping[str, torch.Tensor],
        x_t: torch.Tensor,
        time: torch.Tensor,
    ) -> torch.Tensor:
        return x_t + inputs["state"].mean(dim=-1, keepdim=True).view(-1, 1, 1)


toy = ToyAdapter()
inputs = {"state": torch.tensor([[1.0, 1.0], [2.0, 2.0]])}
delta = state_condition_delta(toy, inputs, torch.zeros(2, 3, 1), torch.tensor([0.2, 0.8]))
assert torch.isclose(delta, torch.tensor(1.5))
print("state-condition delta:", float(delta))

**Explain (2–3 sentences).** Why must the cached prefix be rebuilt after the state or image changes? Which parts are held fixed in the probe?

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POLICY, PREPROCESSOR, POSTPROCESSOR = load_prepared_smolvla("published", DATA, device=DEVICE)
ACTION_CHUNK_SIZE = POLICY.config.chunk_size
DATASET = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "validation",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
RAW = default_collate([DATASET[0]])
RAW["task"] = ["put the bowl on the plate"]
BATCH = PREPROCESSOR(RAW)
BATCH = {
    key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
    for key, value in BATCH.items()
}
ADAPTER = SmolVLAAdapter(POLICY)
INPUTS = ADAPTER.prepare_inputs(BATCH)
NOISE = torch.randn(1, ADAPTER.chunk_size, ADAPTER.max_action_dim, device=DEVICE)
TIME = torch.tensor([0.5], device=DEVICE)
DELTA = state_condition_delta(ADAPTER, INPUTS, NOISE, TIME)
PREFIX_PAD, PREFIX_CACHE = ADAPTER.prefix_cache(INPUTS)
assert PREFIX_CACHE is not None
CACHE_LAYERS = PREFIX_CACHE.layers
assert len(CACHE_LAYERS) > 0 and CACHE_LAYERS[0].keys is not None
print("mean absolute velocity change after zeroing state:", float(DELTA))
print(
    "prefix padding shape:",
    tuple(PREFIX_PAD.shape),
    "| cached layers:",
    len(CACHE_LAYERS),
)
print("first key tensor shape:", tuple(CACHE_LAYERS[0].keys.shape))

This compares one condition while keeping the rest fixed. The plot below is a visual toy model, not the SmolVLA policy and not a training cell.